# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.co

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [12]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Blog Posts, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [13]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [14]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [15]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [16]:
select_relevant_links("https://edwarddonner.com")

KeyboardInterrupt: 

In [17]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [18]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 7 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'blog posts', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2025/11/11/ai-live-event/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/'},
  {'type': 'company homepage', 'url': 'https://edwarddonner.com/'}]}

In [20]:
select_relevant_links("https://www.cyberdyne.com/")

Selecting relevant links for https://www.cyberdyne.com/ by calling gpt-5-nano
Found 6 relevant links


{'links': [{'type': 'company page', 'url': 'https://www.cyberdyne.com'},
  {'type': 'about page',
   'url': 'https://cyberdyne.jp/english/company/Information.html'},
  {'type': 'blog post',
   'url': 'https://www.cyberdyne.com/global-blog/%e3%80%90news%e3%80%91delegation-from-national-taiwan-university-hospital-ntuh-visits-cyberdyne/'},
  {'type': 'blog post',
   'url': 'https://www.cyberdyne.com/global-blog/%e3%80%90news%e3%80%91paper-on-cybernics-treatment-published-in-the-journal-regenerative-medicine-of-the-japanese-society-for-regenerative-medicine/'},
  {'type': 'blog post',
   'url': 'https://www.cyberdyne.com/global-blog/%e3%80%90news%e3%80%91cyberdyne-agrees-on-comprehensive-collaboration-with-universiti-malaysia-perlis-toward-social-implementation-of-hcps-integrated-cybernics-technology/'},
  {'type': 'blog post',
   'url': 'https://www.cyberdyne.com/global-blog/%e3%80%90news%e3%80%91cyberdyne-signs-strategic-mou-with-carnegie-mellon-university-cmu-a-global-leader-in-ai-and-r

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [21]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [22]:
print(fetch_page_and_all_relevant_links("https://www.cyberdyne.com/"))

Selecting relevant links for https://www.cyberdyne.com/ by calling gpt-5-nano
Found 14 relevant links
## Landing Page:

Cyberdyne Inc.

About us
Contact us
Contact Us
Menu
Cybernics: the foundation of the business
Cybernics is a new academic field that fused/combined cross-disciplinary fields. The core disciplines of Cybernics are academic fields related to humans, robots, and information systems, such as neuroscience, AI, Robotics, system engineering, information technology, physiology, psychology, behavioral science, philosophy, ethics, law, business administration, and many more.
Complex problems that we face today are difficult to solve from a single direction. As a new academic field that approaches such issues from multiple directions, Yoshiyuki Sankai, a professor at the University of Tsukuba in Japan (the “Company”), founded Cybernics.
To disseminate innovation needed for society, Sankai recognized the need to work closely with practical fields. Sankai founded CYBERDYNE Inc. in

In [38]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [25]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [26]:
# company_name = "CYBERDYNE"
# company_url = "https://www.cyberdyne.com/"
company_name = "HuggingFace"
company_url = "https://huggingface.co"
get_brochure_user_prompt(company_name, company_url)

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 17 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nHauhauCS/Qwen3.5-35B-A3B-Uncensored-HauhauCS-Aggressive\nUpdated\n12 days ago\n•\n276k\n•\n809\nJackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled\nUpdated\n2 days ago\n•\n141k\n•\n1.04k\nmistralai/Mistral-Small-4-119B-2603\nUpdated\n6 days ago\n•\n10.3k\n•\n297\nbaidu/Qianfan-OCR\nUpdated\n4 days ago\n•\n5.48k\n•\n297\nfishaudio/s2-pro\nUpdated\n12 days ago\n•\n12.3k\n•\n710\nBrow

In [28]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [30]:
# company_name = "CYBERDYNE"
# company_url = "https://www.cyberdyne.com/"
company_name = "HuggingFace"
company_url = "https://huggingface.co"
create_brochure(company_name, company_url)


Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is the AI community building the future of machine learning. It serves as a vibrant collaboration platform where the global machine learning (ML) community creates, discovers, and shares models, datasets, and applications. Hugging Face hosts over 2 million models, 500,000+ datasets, and 1 million+ applications, empowering developers, researchers, and organizations to accelerate their ML projects across various modalities—including text, image, video, audio, and even 3D.

---

## What We Offer

- **Hugging Face Hub:** A central place to host, share, and collaborate on unlimited public machine learning models and datasets.
- **Open-Source Stack:** Tools and frameworks designed to help the ML community move faster, experiment boldly, and build with confidence.
- **Applications & Spaces:** A thriving ecosystem that supports deploying AI apps and interactive spaces, allowing users to bring models to life and showcase their innovations.
- **Diverse Modalities:** Support for text, image, video, audio, and 3D data enables cutting-edge research and applications in multiple AI domains.
- **Enterprise Solutions:** Scalable, secure offerings tailored for business needs, helping organizations integrate and deploy AI solutions effectively.

---

## Community and Culture

At its core, Hugging Face is a **collaborative open-source community** committed to building an **open and ethical AI future**. The platform empowers ML engineers, data scientists, and AI practitioners from around the world to:

- Share and discover cutting-edge models and datasets
- Collaborate openly with peers to push forward AI research and applications
- Build their portfolios and gain visibility in the growing ML ecosystem

Hugging Face's culture is centered on **innovation, transparency, and inclusivity**, fueling the rapid pace of AI development and adoption globally.

---

## Customers and Use Cases

Hugging Face caters to a wide range of users including:

- **Researchers and Academics:** Access and contribute state-of-the-art models and datasets freely.
- **Startups and Developers:** Leverage powerful open-source tools and pre-trained models to build new AI-powered products quickly.
- **Large Enterprises:** Implement scalable solutions for text, vision, speech, and beyond with enterprise-grade support.
- **End Users:** Explore AI apps and communities to engage with AI-driven services and experiments.

Some key industries benefiting from Hugging Face include healthcare, finance, technology, media, and education.

---

## Careers at Hugging Face

Join a passionate team dedicated to transforming AI development worldwide.

- Be part of an **innovative, open-source driven environment** shaping the future of machine learning.
- Collaborate daily with a **diverse global community** of experts and contributors.
- Work on **cutting-edge AI technologies** with real-world impact.
- Enjoy a culture embracing **transparency, inclusivity, and continuous learning**.

Explore opportunities to contribute in engineering, research, product management, community engagement, and more.

---

## Get Started with Hugging Face

- **Browse 2M+ Models:** Discover trending and specialized models updated regularly.
- **Explore 500K+ Datasets:** Find rich datasets spanning multiple domains.
- **Build and Share AI Apps:** Use Spaces to deploy interactive ML applications effortlessly.
- **Join the Community:** Sign up to build your ML profile and contribute to an open, ethical AI future.

Visit [huggingface.co](https://huggingface.co) and be part of the machine learning revolution.

---

### Brand Identity

- Signature Colors: Bright Yellow (#FFD21E), Orange (#FF9D00), and Gray (#6B7280)
- Logo and assets are designed for clear, friendly, and approachable communication in AI.

---

*Hugging Face — The AI community building the future.*

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [34]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [35]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 14 relevant links


# Welcome to Hugging Face – Where AI Gets Friendly 🤗

---

## Who Are We?

Hugging Face is *THE* AI community that’s not just building the future — it’s collaborating to create it. Think of us as the bustling town square where thousands of machine learning enthusiasts, engineers, and data scientists gather to share models, datasets, and cool AI applications. We’re open, ethical, and unapologetically passionate about making AI accessible and impactful for everyone.

---

## What’s the Buzz?

- **2 Million+ Models:** From Qwen3.5-35B to OCR wizards, our hub hosts more models than you have tabs open on your browser.
- **500,000+ Datasets:** Data is the new oil and we’ve got barrels of it, ready for your next big ML project.
- **1 Million+ Applications:** Apps that translate text, generate videos, even turn images into memes faster than your last coffee break.
- **Spaces & Buckets:** Sandbox environments to show off your own ML masterpieces or run AI experiments — no PhD required.

---

## Culture: *Collaboration*, *Innovation*, and a Dash of Fun

- **Community First:** We believe the best AI happens when brains (and hearts) unite globally.
- **Open Source & Ethical:** Transparency is our middle name — well, not literally, but you get the point.
- **Accelerate Your ML Journey:** Whether you're a newbie or a ninja, we’re here with open-source stacks to make you faster, smarter, and ready to change the world.
- **We’re Friendly:** The “Hugging Face” isn’t just a name — it’s a vibe. Here, you’re encouraged to be curious, crazy, and collaborative.

---

## Careers: Join the AI Party 🎉

Ready to build the future with a passionate crew pushing AI boundaries every day? Hugging Face offers thrilling opportunities where your code (and memes) will be appreciated. We’re hunting for:

- Machine Learning Engineers & Scientists
- Open Source Advocates
- AI Application Developers
- Community Builders & Evangelists

If you want your day-to-day to look like sci-fi, but with real impact, and a dash of humor, come join the crew!

---

## Why Hugging Face?

- **Explore & Share:** Upload your own models, show off your datasets, or dive into the treasure trove others have left behind.
- **Build Your ML Portfolio:** Create, host, and collaborate — build your rep in the AI world.
- **Multimodal Magic:** Text, images, audio, video, even 3D — we play with all the toys.
- **Enterprise & Docs:** Top-notch tools and support for individuals and large teams alike.
- **Community-Powered:** Every day, thousands of updates, new models, datasets, and applications keep our hug-o-meter buzzing.

---

## Take a Hug from the Future 🤗

Why settle for AI that’s cold and confusing when you can join a community that’s collaborative, ethical, and downright fun? Whether you want to explore trends, contribute, or build the next AI superstar app, Hugging Face is your go-to hub for everything machine-learning-flavored.

---

**Ready to dive in?**  
[Sign Up Now] — Your AI journey just got friendlier.

---

**Contact Us:**  
Website: huggingface.co  
Stay curious. Stay collaborative. Hug AI right.

In [39]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:
# company_name = "CYBERDYNE"
# company_url = "https://www.cyberdyne.com/"

stream_brochure("CYBERDYNE", "https://www.cyberdyne.com/")

Selecting relevant links for https://www.cyberdyne.com/ by calling gpt-5-nano
Found 13 relevant links


# Welcome to CYBERDYNE Inc.  
*Empowering humankind – one robot at a time!*  

---

## Who We Are  
CYBERDYNE isn’t just another tech company; we are the pioneers of **Cybernics**, a radical new field blending humans + robots + information systems with a dash of neuroscience, AI, physiology, psychology, ethics, philosophy, and even a sprinkle of business savvy. Founded in 2004 by Professor Yoshiyuki Sankai of the University of Tsukuba, Japan, our company's name is a power-packed combo of **CYBERnics + Dyne (power)** — because we don’t just build tech, we empower *you*.  

By fusing the human body with cyber/physical spaces, we’re not doing sci-fi—we’re doing real innovation that helps people walk again, communicate again, and live better lives.  

---

## What We Do: Robots That Really Get You Moving  

### Medical HAL Series  
Meet your new robotic sidekick designed to help restore your natural movement and strength:  
- **HAL Lower Limb Type**: Helping patients reclaim their ability to walk—think of it as a power-up for your legs!  
- **HAL Lumbar Type**: Supports both caregivers and those cared for, easing back pain and boosting mobility, because every hero needs a strong back.  
- **HAL Single Joint Type**: Focused rehab for elbows, knees, and ankles—the robotics equivalent of a personal trainer.  

### Autonomous Robots: Keeping it Clean & Moving Heavy  
- **Cleaning Robot**: Taking cleaning and disinfection to a whole new level of autonomous awesome. Think of it as your ultra-efficient, tireless cleaning ninja.  
- **Transportation Robot**: Carries heavy loads without complaining—your robotic best friend for the heavy lifting.  

### Communication Help for Those Who Need it Most  
- **Cyin for Living Support** and **Acoustic X**: Innovative interfaces making communication possible for patients who have severe difficulties speaking or moving.  
Because everyone deserves a voice.  

---

## Our Customers: Those Who Dare to Move Forward  
From patients recovering mobility, caregivers seeking support, healthcare providers and researchers pushing the boundaries of rehab technology, to institutions like the National Taiwan University Hospital who come knocking to learn about our breakthroughs – CYBERDYNE serves those who believe in the power of medicine, robotics, and human determination. Our tech is designed for:  
- Spinal cord injury and stroke patients  
- Seniors wanting to regain freedom  
- Athletes needing extra joint support  
- Laborers with back strain  
…and anyone ready to embrace the future of human-machine synergy!  

---

## Life at CYBERDYNE: Join Our Robo-Team!  

### Culture & Careers  
Do you want to work where philosophy meets AI? Where rehabilitation robotics meet practical application? CYBERDYNE offers a workplace where diverse disciplines collide—from neuroscience and ethics to engineering and business—to solve complex problems through team-based innovation.  

If you thrive on cutting-edge technology, enjoy cross-disciplinary collaboration, and want to make a global impact, come join us. Your job will be anything but ordinary—because ordinary doesn’t fix paralysis or rewrite the future of human ability.  

Current openings include roles in robotics engineering, AI research, system integration, clinical support, and more.  

---

## Why CYBERDYNE?  
Because when robots meet humans, magic happens. We don’t just dream about the future—we build it. By combining the brilliance of multiple disciplines, the drive for social innovation, and the heart of humanity, Cyberdyne is the place where science fiction becomes science fact.  

---

**Stay Connected**  
Curious about our latest breakthroughs or want to hear real stories of human & robotic teamwork? Check out our Newsroom for the latest buzz.  

---

### CYBERDYNE Inc.  
*Where Cybernics power up your potential.*  

**Want to learn more or join us?** Visit [cyberdyne.jp](http://cyberdyne.jp) or reach out—because the future doesn’t wait, and neither should you.  

---

*P.S. HAL isn’t just a robot; it’s your new best friend in the journey back to movement.* 🤖❤️

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>